# **Dataset Maker**

Follow this notebook to transform your data into a dataset that can be used to train the model.

## 📋 **1. Required Data**

RNeNcodec requires three types of data: **audio files**, **annotations**, and **parameter specifications**. Each one is explained below.

### **1.1 🎵 Audio Files**

Most audio file formats and sampling rates are supported, although the final model will always operate at 24kHz.

**Supported Audio Formats**: `.wav`, `.mp3`, `.flac`, `.aac`, `.ogg`, `.m4a`, `.wma`, `.aiff`, `.au`, `.ra`, `.3gp`, `.amr`, `.ac3`, `.dts`, `.ape`, `.mka`, `.opus`

### **1.2 📊 Annotations (CSV Files)**

Each audio file **must** have a corresponding CSV file with the **exact same base name**:

| Audio File | CSV File | Status |
|------------|----------|---------|
| `piano.wav` | `piano.csv` | ✅ Correct |
| `flute.mp3` | `flute_annotations.csv` | ⛔ Incorrect - name mismatch! |

The CSV annotation files must follow this specific structure:

- **Header row** with parameter names (must match those in `parameters.json`)
- **One row per frame** (75 frames per second)
- **Continuous parameters**: Numeric values (will be normalized to [0,1] range)
- **Categorical parameters**: Non-negative integers representing each class (0, 1, 2, ...)

Example CSV:

| saturation | reverb | instrument |
|------------|--------|------------|
| 10.5       | 40.3   | 0          |
| 10.0       | 40.32  | 1          |
| 9.8        | 40.25  | 0          |
| ...        | ...    | ...        |

### **1.3 ⚙️ Parameter Specifications**

Information about the parameters to be controlled must be stored in a `parameters.json` file. Each parameter must specify its **name** and **type** (either `continuous` or `class`). Additional features will depepnd on the type of the parameter as follow:

**🔢 Continuous Parameters** (numeric values like tempo, volume, saturation):
- `min`/`max`: Range used for normalization
- `unit`: Physical unit (e.g., bpm, %, dB)

**🏷️ Class Parameters** (categorical values like instrument type, genre):
- `classes`: Ordered list of class names (order determines integer mapping)

#### Example `parameters.json`:

```json
{
    "parameter_1": {"name": "saturation", "type": "continuous", "unit": "dB", "min": 0,   "max": 12}, 
    "parameter_2": {"name": "reverb",     "type": "continuous", "unit": "%",  "min": 0,   "max": 100},
    "parameter_3": {"name": "instrument", "type": "class",      "classes": ["piano", "flute"]}
}
```
## **📁 2. Required Data Structure**

Before running this notebook on your data, make sure it is structured as follows:

**Option 1:** Simple Structure (all data together)

```
dataset_folder/
└── raw/                         # Raw input data
    ├── parameters.json          # Parameter configuration file
    ├── piano.wav                # Audio files (various formats supported)
    ├── piano.csv                # Corresponding CSV annotations
    ├── flute.mp3                # More audio files...
    ├── flute.csv                # More CSV files...
    └── ...
```

**Option 2:** Split Structure (separate train/validation/test sets)
If you want to use specific data splits for training, validation, and testing.

```
dataset_folder/
└── raw/                         # Raw input data
    ├── parameters.json          # Parameter configuration file
    ├── train/
    │   ├── piano.wav            # Training audio files
    │   ├── piano.csv            # Training CSV annotations
    │   └── ...
    ├── validation/
    │   ├── flute.mp3            # Validation audio files
    │   ├── flute.csv            # Validation CSV files
    │   └── ...
    └── test/
        ├── violin.mp3           # Test audio files
        ├── violin.csv           # Test CSV files
        └── ...
```

## **🔄 3. Dataset Creation Pipeline**

Once your data is properly arranged, follow this notebook to create your dataset. The full pipeline consists of five steps:

- **📊 Visualization** - Analyze your raw data structure and parameters to verify everything is in order
- **🔧 Normalization** - Standardize format and normalize your data
- **🎵 EnCodec Encoding** - Convert audio to a compressed latent format that RNeNcodec can process
- **📄 Sidecar Creation**- Align parameters with audio frames (75 fps)
- **🤗 HuggingFace Dataset** - Package everything into a training-ready format

Let's get started! 👇

In [1]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent if (pathlib.Path.cwd().name == "rnencodec_notebooks") else pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

datasets_paths = [
    "../trumpet_frequency",
    "../trumpet_class",
    "../trumpet_fourier",
    "../trumpet_log_normalised",
    "../trumpet_sine_cosine",
]

### **3.1 📊 Visualization**

Visualize your data and make sure everything is alright.

In [2]:
# Import visualization functions
from dataprep.step_0_visualization import quick_analyze

# 🔍 ANALYZE ENTIRE DATASET
for dataset_path in datasets_paths:
    quick_analyze(dataset_path, individual_plot_selector=True) # Set individual_plot_selector=True to choose which files to plot.

DATASET SUMMARY:

✅ Found 49 audio-CSV pairs
📁 Parameters: pitch
⏱️ Total audio duration: 702.9 seconds

FILE DETAILS:

✅ 280-1: 19.0s, 1425 frames
✅ 315-1: 14.0s, 1050 frames
⚠️ 316-4: 14.4s, 1080 frames (1 extra audio frames in audio file)
⚠️ 268-1: 18.5s, 1387 frames (1 extra audio frames in audio file)
✅ 375-3: 18.7s, 1400 frames
✅ 335-3: 10.7s, 800 frames
✅ 257-1: 19.0s, 1425 frames
⚠️ 234-1: 6.5s, 487 frames (1 extra audio frames in audio file)
✅ 317-2: 8.7s, 650 frames
✅ 440-3: 13.3s, 1000 frames
✅ 442-1: 22.7s, 1700 frames
✅ 418-1: 8.0s, 600 frames
⚠️ 375-2: 18.4s, 1379 frames (1 extra audio frames in audio file)
⚠️ 396-3: 8.6s, 647 frames (1 extra audio frames in audio file)
✅ 302-2: 17.7s, 1325 frames
✅ 356-1: 21.3s, 1600 frames
✅ 333-2: 12.0s, 900 frames
⚠️ 280-3: 13.8s, 1038 frames (1 extra audio frames in audio file)
✅ 334-1: 5.3s, 400 frames
✅ 446-2: 10.7s, 800 frames
✅ 300-1: 21.0s, 1575 frames
✅ 420-3: 12.0s, 900 frames
✅ 354-3: 14.0s, 1050 frames
✅ 317-3: 19.3s, 1450 f

DATASET SUMMARY:

✅ Found 49 audio-CSV pairs
📁 Parameters: pitch_class
⏱️ Total audio duration: 702.9 seconds

FILE DETAILS:

✅ 280-1: 19.0s, 1425 frames
✅ 315-1: 14.0s, 1050 frames
⚠️ 316-4: 14.4s, 1080 frames (1 extra audio frames in audio file)
⚠️ 268-1: 18.5s, 1387 frames (1 extra audio frames in audio file)
✅ 375-3: 18.7s, 1400 frames
✅ 335-3: 10.7s, 800 frames
✅ 257-1: 19.0s, 1425 frames
⚠️ 234-1: 6.5s, 487 frames (1 extra audio frames in audio file)
✅ 317-2: 8.7s, 650 frames
✅ 440-3: 13.3s, 1000 frames
✅ 442-1: 22.7s, 1700 frames
✅ 418-1: 8.0s, 600 frames
⚠️ 375-2: 18.4s, 1379 frames (1 extra audio frames in audio file)
⚠️ 396-3: 8.6s, 647 frames (1 extra audio frames in audio file)
✅ 302-2: 17.7s, 1325 frames
✅ 356-1: 21.3s, 1600 frames
✅ 333-2: 12.0s, 900 frames
⚠️ 280-3: 13.8s, 1038 frames (1 extra audio frames in audio file)
✅ 334-1: 5.3s, 400 frames
✅ 446-2: 10.7s, 800 frames
✅ 300-1: 21.0s, 1575 frames
✅ 420-3: 12.0s, 900 frames
✅ 354-3: 14.0s, 1050 frames
✅ 317-3: 19.3s, 

DATASET SUMMARY:

✅ Found 49 audio-CSV pairs
📁 Parameters: pitch_sin_1, pitch_cos_1, pitch_sin_2, pitch_cos_2, pitch_sin_3, pitch_cos_3, pitch_sin_4, pitch_cos_4
⏱️ Total audio duration: 702.9 seconds

FILE DETAILS:

✅ 280-1: 19.0s, 1425 frames
✅ 315-1: 14.0s, 1050 frames
⚠️ 316-4: 14.4s, 1080 frames (1 extra audio frames in audio file)
⚠️ 268-1: 18.5s, 1387 frames (1 extra audio frames in audio file)
✅ 375-3: 18.7s, 1400 frames
✅ 335-3: 10.7s, 800 frames
✅ 257-1: 19.0s, 1425 frames
⚠️ 234-1: 6.5s, 487 frames (1 extra audio frames in audio file)
✅ 317-2: 8.7s, 650 frames
✅ 440-3: 13.3s, 1000 frames
✅ 442-1: 22.7s, 1700 frames
✅ 418-1: 8.0s, 600 frames
⚠️ 375-2: 18.4s, 1379 frames (1 extra audio frames in audio file)
⚠️ 396-3: 8.6s, 647 frames (1 extra audio frames in audio file)
✅ 302-2: 17.7s, 1325 frames
✅ 356-1: 21.3s, 1600 frames
✅ 333-2: 12.0s, 900 frames
⚠️ 280-3: 13.8s, 1038 frames (1 extra audio frames in audio file)
✅ 334-1: 5.3s, 400 frames
✅ 446-2: 10.7s, 800 frames
✅ 300-1:

DATASET SUMMARY:

✅ Found 49 audio-CSV pairs
📁 Parameters: pitch_log_norm
⏱️ Total audio duration: 702.9 seconds

FILE DETAILS:

✅ 280-1: 19.0s, 1425 frames
✅ 315-1: 14.0s, 1050 frames
⚠️ 316-4: 14.4s, 1080 frames (1 extra audio frames in audio file)
⚠️ 268-1: 18.5s, 1387 frames (1 extra audio frames in audio file)
✅ 375-3: 18.7s, 1400 frames
✅ 335-3: 10.7s, 800 frames
✅ 257-1: 19.0s, 1425 frames
⚠️ 234-1: 6.5s, 487 frames (1 extra audio frames in audio file)
✅ 317-2: 8.7s, 650 frames
✅ 440-3: 13.3s, 1000 frames
✅ 442-1: 22.7s, 1700 frames
✅ 418-1: 8.0s, 600 frames
⚠️ 375-2: 18.4s, 1379 frames (1 extra audio frames in audio file)
⚠️ 396-3: 8.6s, 647 frames (1 extra audio frames in audio file)
✅ 302-2: 17.7s, 1325 frames
✅ 356-1: 21.3s, 1600 frames
✅ 333-2: 12.0s, 900 frames
⚠️ 280-3: 13.8s, 1038 frames (1 extra audio frames in audio file)
✅ 334-1: 5.3s, 400 frames
✅ 446-2: 10.7s, 800 frames
✅ 300-1: 21.0s, 1575 frames
✅ 420-3: 12.0s, 900 frames
✅ 354-3: 14.0s, 1050 frames
✅ 317-3: 19.3

DATASET SUMMARY:

✅ Found 49 audio-CSV pairs
📁 Parameters: pitch_sin, pitch_cos
⏱️ Total audio duration: 702.9 seconds

FILE DETAILS:

✅ 280-1: 19.0s, 1425 frames
✅ 315-1: 14.0s, 1050 frames
⚠️ 316-4: 14.4s, 1080 frames (1 extra audio frames in audio file)
⚠️ 268-1: 18.5s, 1387 frames (1 extra audio frames in audio file)
✅ 375-3: 18.7s, 1400 frames
✅ 335-3: 10.7s, 800 frames
✅ 257-1: 19.0s, 1425 frames
⚠️ 234-1: 6.5s, 487 frames (1 extra audio frames in audio file)
✅ 317-2: 8.7s, 650 frames
✅ 440-3: 13.3s, 1000 frames
✅ 442-1: 22.7s, 1700 frames
✅ 418-1: 8.0s, 600 frames
⚠️ 375-2: 18.4s, 1379 frames (1 extra audio frames in audio file)
⚠️ 396-3: 8.6s, 647 frames (1 extra audio frames in audio file)
✅ 302-2: 17.7s, 1325 frames
✅ 356-1: 21.3s, 1600 frames
✅ 333-2: 12.0s, 900 frames
⚠️ 280-3: 13.8s, 1038 frames (1 extra audio frames in audio file)
✅ 334-1: 5.3s, 400 frames
✅ 446-2: 10.7s, 800 frames
✅ 300-1: 21.0s, 1575 frames
✅ 420-3: 12.0s, 900 frames
✅ 354-3: 14.0s, 1050 frames
✅ 317-3

### **3.2 🔧 Normalization**

This step prepares your audio files by:
1. **Resampling** all audio to 24kHz mono (required by EnCodec)
2. **RMS Normalization** (optional) to ensure consistent loudness across your dataset (you may want apply_rms_normalization=False if the relative volume between your dataset files is important).

In [3]:
from dataprep.step_1_normalization import quick_normalize

for dataset_path in datasets_paths:
    quick_normalize(dataset_path, apply_rms_normalization=False)


DATA SUMMARY:

Found 49 audio-CSV pairs
RMS normalization: DISABLED (resampling to 24kHz only)
Input: ../trumpet_frequency/raw
Output: ../trumpet_frequency/normalized

DATA PROCESSING:

Processing: 234-1.wav
    RMS normalization disabled - resampling to 24kHz only
    🔄 Trimmed audio: 488 → 487 frames (1 extra audio frames in audio file)
    ✓ Saved: 234-1.wav
    ✅ Copied CSV: 234-1.csv → ../trumpet_frequency/normalized/234-1.csv

Processing: 236-2.wav
    RMS normalization disabled - resampling to 24kHz only
    🔄 Trimmed audio: 1260 → 1259 frames (1 extra audio frames in audio file)
    ✓ Saved: 236-2.wav
    ✅ Copied CSV: 236-2.csv → ../trumpet_frequency/normalized/236-2.csv

Processing: 236-3.wav
    RMS normalization disabled - resampling to 24kHz only
    ✓ Saved: 236-3.wav
    ✅ Copied CSV: 236-3.csv → ../trumpet_frequency/normalized/236-3.csv

Processing: 237-4.wav
    RMS normalization disabled - resampling to 24kHz only
    🔄 Trimmed audio: 1388 → 1387 frames (1 extra audi

### **3.3 🎵 EnCodec Encoding**

In [4]:
from dataprep.step_2_encodec import quick_encode

for dataset_path in datasets_paths:
    quick_encode(dataset_path, bandwidth=6.0, device="cpu")  # Force 8 codebooks and CPU


DATA PROCESSING:

Found 49 WAV files under ../trumpet_frequency/normalized
Using device: cpu
Loading EnCodec model (facebook/encodec_24khz)...


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

The image processor of type `EncodecImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`use_fast` is set to `True` but the image processor class does not have a fast version.  Falling back to the slow version.
Encoding: 100%|██████████| 49/49 [00:00<00:00, 17757.12file/s]



DATA SUMMARY:

📁 234-1.ecdc
   Audio codes shape: (1, 1, 8, 487)
   Frames: unknown
   Codebooks: unknown
   Original length: 155840 samples

📁 236-2.ecdc
   Audio codes shape: (1, 1, 8, 1259)
   Frames: unknown
   Codebooks: unknown
   Original length: 402879 samples

📁 236-3.ecdc
   Audio codes shape: (1, 1, 8, 1275)
   Frames: unknown
   Codebooks: unknown
   Original length: 407989 samples

📁 237-4.ecdc
   Audio codes shape: (1, 1, 8, 1387)
   Frames: unknown
   Codebooks: unknown
   Original length: 443840 samples

📁 255-2.ecdc
   Audio codes shape: (1, 1, 8, 693)
   Frames: unknown
   Codebooks: unknown
   Original length: 221760 samples

📁 255-4.ecdc
   Audio codes shape: (1, 1, 8, 1312)
   Frames: unknown
   Codebooks: unknown
   Original length: 419840 samples

📁 257-1.ecdc
   Audio codes shape: (1, 1, 8, 1425)
   Frames: unknown
   Codebooks: unknown
   Original length: 455989 samples

📁 257-3.ecdc
   Audio codes shape: (1, 1, 8, 1125)
   Frames: unknown
   Codebooks: unknow

Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

...

Encoding: 100%|██████████| 49/49 [00:00<00:00, 20547.98file/s]



DATA SUMMARY:

📁 234-1.ecdc
   Audio codes shape: (1, 1, 8, 487)
   Frames: unknown
   Codebooks: unknown
   Original length: 155840 samples

📁 236-2.ecdc
   Audio codes shape: (1, 1, 8, 1259)
   Frames: unknown
   Codebooks: unknown
   Original length: 402879 samples

📁 236-3.ecdc
   Audio codes shape: (1, 1, 8, 1275)
   Frames: unknown
   Codebooks: unknown
   Original length: 407989 samples

📁 237-4.ecdc
   Audio codes shape: (1, 1, 8, 1387)
   Frames: unknown
   Codebooks: unknown
   Original length: 443840 samples

📁 255-2.ecdc
   Audio codes shape: (1, 1, 8, 693)
   Frames: unknown
   Codebooks: unknown
   Original length: 221760 samples

📁 255-4.ecdc
   Audio codes shape: (1, 1, 8, 1312)
   Frames: unknown
   Codebooks: unknown
   Original length: 419840 samples

📁 257-1.ecdc
   Audio codes shape: (1, 1, 8, 1425)
   Frames: unknown
   Codebooks: unknown
   Original length: 455989 samples

📁 257-3.ecdc
   Audio codes shape: (1, 1, 8, 1125)
   Frames: unknown
   Codebooks: unknow

Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

Encoding: 100%|██████████| 49/49 [00:00<00:00, 17772.47file/s]


DATA SUMMARY:

📁 234-1.ecdc
   Audio codes shape: (1, 1, 8, 487)
   Frames: unknown
   Codebooks: unknown
   Original length: 155840 samples

📁 236-2.ecdc
   Audio codes shape: (1, 1, 8, 1259)
   Frames: unknown
   Codebooks: unknown
   Original length: 402879 samples

📁 236-3.ecdc
   Audio codes shape: (1, 1, 8, 1275)
   Frames: unknown
   Codebooks: unknown
   Original length: 407989 samples

📁 237-4.ecdc
   Audio codes shape: (1, 1, 8, 1387)
   Frames: unknown
   Codebooks: unknown
   Original length: 443840 samples

📁 255-2.ecdc
   Audio codes shape: (1, 1, 8, 693)
   Frames: unknown
   Codebooks: unknown
   Original length: 221760 samples

📁 255-4.ecdc
   Audio codes shape: (1, 1, 8, 1312)
   Frames: unknown
   Codebooks: unknown
   Original length: 419840 samples

📁 257-1.ecdc
   Audio codes shape: (1, 1, 8, 1425)
   Frames: unknown
   Codebooks: unknown
   Original length: 455989 samples

📁 257-3.ecdc
   Audio codes shape: (1, 1, 8, 1125)
   Frames: unknown
   Codebooks: unknow

### **3.4 📄 Sidecar Creation**

In [5]:
from dataprep.step_3_sidecars import quick_create_sidecars

for dataset_path in datasets_paths:
    results = quick_create_sidecars(dataset_path)


SIDECAR CREATION:

📁 Tokens directory: ../trumpet_frequency/tokens
📁 Raw directory: ../trumpet_frequency/raw
📊 Found 49 .ecdc-CSV pairs
🎛️ Parameters: pitch

SIDECARS SUMMARY:

📄 335-3 sidecar:
    EnCodec frames: 800
    CSV annotations: 800
    ✅ Perfect alignment!
    ✅ Sidecar created (800 frames, 1 features)

📄 446-2 sidecar:
    EnCodec frames: 800
    CSV annotations: 800
    ✅ Perfect alignment!
    ✅ Sidecar created (800 frames, 1 features)

📄 269-2 sidecar:
    EnCodec frames: 1800
    CSV annotations: 1800
    ✅ Perfect alignment!
    ✅ Sidecar created (1800 frames, 1 features)

📄 334-1 sidecar:
    EnCodec frames: 400
    CSV annotations: 400
    ✅ Perfect alignment!
    ✅ Sidecar created (400 frames, 1 features)

📄 396-3 sidecar:
    EnCodec frames: 647
    CSV annotations: 647
    ✅ Perfect alignment!
    ✅ Sidecar created (647 frames, 1 features)

📄 282-2 sidecar:
    EnCodec frames: 1162
    CSV annotations: 1162
    ✅ Perfect alignment!
    ✅ Sidecar created (1162 fra

### **3.5 🤗 HuggingFace Dataset**
(on Windows, you can ignore the "Failed simlink" messages)

In [6]:
from dataprep.step_4_HF import quick_create_dataset

for dataset_path in datasets_paths:
    results = quick_create_dataset(dataset_path)


HUGGINGFACE DATASET CREATION:

📁 Tokens directory: ../trumpet_frequency/tokens
📁 Output directory: ../trumpet_frequency/hf_dataset
🔗 Materialize mode: link
🧹 Cleaning up dataset directory: ../trumpet_frequency/hf_dataset/tokens
✅ Cleaned up tokens directory

📊 No split structure detected
📌 Using all data as 'train' split

   📂 train: 49 .ecdc files found

📋 Saved expanded conditioning config to: conditioning_config.json
   • Features: pitch
   • Total features: 1

💾 Saved DatasetDict to: ../trumpet_frequency/hf_dataset
📈 Dataset summary:
   • train: 49 samples

🔍 Verifying dataset files...
✅ All audio files verified successfully

HUGGINGFACE DATASET CREATION:

📁 Tokens directory: ../trumpet_class/tokens
📁 Output directory: ../trumpet_class/hf_dataset
🔗 Materialize mode: link
🧹 Cleaning up dataset directory: ../trumpet_class/hf_dataset/tokens
✅ Cleaned up tokens directory

📊 No split structure detected
📌 Using all data as 'train' split

   📂 train: 49 .ecdc files found

📋 Saved expande